In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 69
==================================================

Week: 10 of 24
Day: 69 of 168
Date: Saturday, January 10, 2026
Topic: LunarLander Environment Setup & Training

Week 10 Progress:
✅ Day 64: RL Fundamentals - MDP, Value Functions, Bellman Equations (COMPLETED)
✅ Day 65: Q-Learning & Temporal Difference Learning (COMPLETED)
✅ Day 66: Deep Q-Networks (DQN) Theory & Architecture (COMPLETED)
✅ Day 67: DQN Improvements & Variants (COMPLETED)
✅ Day 68: DQN Training & Optimization Techniques (COMPLETED)
🔄 Day 69: LunarLander Environment Setup & Training (TODAY!)
⬜ Day 70: LunarLander DQN Training & Week Completion

Progress: 85.7% (6/7 days)

==================================================
🎯 Week 10 Project: Solve LunarLander with DQN
- Apply all techniques learned this week
- Train optimized DQN agent
- Achieve 200+ average reward
- Complete autonomous lunar landing!

🎯 Today's Learning Objectives:
1. Environment Setup
   - Install gymnasium[box2d]
   - Understand LunarLander environment
   - Analyze state and action spaces
   - Understand reward structure
   - Test environment rendering

2. Baseline Analysis
   - Implement random policy
   - Measure random baseline performance
   - Understand task difficulty
   - Identify key challenges

3. Optimized Agent Implementation
   - Combine all DQN improvements
   - Double DQN + Dueling + Prioritized Replay
   - Configure for LunarLander
   - Set up training infrastructure

4. Initial Training
   - Train for 500+ episodes
   - Monitor progress
   - Save checkpoints
   - Analyze learning curves
   - Adjust if needed

📚 Today's Structure:
Part 1 (1h): Environment Setup & Analysis
Part 2 (1h): Baseline & Understanding Task
Part 3 (2h): Complete Agent Implementation
Part 4 (4h): Training & Monitoring

🎯 SUCCESS CRITERIA:
✅ Environment installed and working
✅ Understand state/action/reward structure
✅ Random baseline measured
✅ Complete agent implemented with all improvements
✅ Training pipeline set up (diagnostics, checkpointing)
✅ Train for 500+ episodes
✅ Reach 0+ average reward (better than random)
✅ Observe improvement trends
✅ Save checkpoints and best model

💡 What We're Applying:
From Day 66: DQN fundamentals
From Day 67: Double + Dueling + Prioritized
From Day 68: LR schedules + Early stopping + Checkpointing

All together → Solve LunarLander! 🚀🌙

==================================================
"""

In [6]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

import sys

print("=" * 80)
print("📦 INSTALLING DEPENDENCIES")
print("=" * 80)

# Install SWIG first (required for Box2D)
print("\n🔄 Installing SWIG...")
!{sys.executable} -m pip install swig -q

# Install Box2D
print("🔄 Installing Box2D...")
!{sys.executable} -m pip install box2d-py -q

# Install gymnasium with Box2D support
print("🔄 Installing gymnasium[box2d]...")
!{sys.executable} -m pip install "gymnasium[box2d]" -q

# Install other required libraries
print("🔄 Installing other dependencies...")
!{sys.executable} -m pip install torch torchvision numpy matplotlib seaborn pandas tqdm opencv-python -q

print("\n✅ All libraries installed!")
print("=" * 80)

# ==================================================
# IMPORT LIBRARIES
# ==================================================

print("\n" + "=" * 80)
print("📚 IMPORTING LIBRARIES")
print("=" * 80)

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque, namedtuple
import random
from tqdm import tqdm
import time
import os
from copy import deepcopy
import datetime

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Gymnasium
import gymnasium as gym

print("✅ Libraries imported!")

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Matplotlib settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Create results directories
os.makedirs('results/plots', exist_ok=True)
os.makedirs('results/models', exist_ok=True)
os.makedirs('results/checkpoints', exist_ok=True)
os.makedirs('results/lunarlander', exist_ok=True)

print("\n✅ Setup complete!")
print("=" * 80)
print(f"\n📊 PyTorch version: {torch.__version__}")
print(f"📊 Gymnasium version: {gym.__version__}")
print(f"📊 Random seed: 42")
print(f"📁 Results directories created!")
print("=" * 80)

print("\n🎯 Ready to explore LunarLander!")
print("=" * 80)

📦 INSTALLING DEPENDENCIES

🔄 Installing SWIG...
🔄 Installing Box2D...


'C:\Program' is not recognized as an internal or external command,
operable program or batch file.


🔄 Installing gymnasium[box2d]...


'C:\Program' is not recognized as an internal or external command,
operable program or batch file.


🔄 Installing other dependencies...


'C:\Program' is not recognized as an internal or external command,
operable program or batch file.



✅ All libraries installed!

📚 IMPORTING LIBRARIES
✅ Libraries imported!

🖥️  Using device: cpu

✅ Setup complete!

📊 PyTorch version: 2.9.1+cpu
📊 Gymnasium version: 1.2.2
📊 Random seed: 42
📁 Results directories created!

🎯 Ready to explore LunarLander!


'C:\Program' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
print("\n" + "=" * 80)
print("🌙 PART 1: LUNARLANDER ENVIRONMENT SETUP & ANALYSIS")
print("=" * 80)


🌙 PART 1: LUNARLANDER ENVIRONMENT SETUP & ANALYSIS


In [8]:
# ==================================================
# EXERCISE 1.1: LUNARLANDER ENVIRONMENT SETUP
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: Setting Up LunarLander Environment")
print("=" * 80)

"""
📖 THEORY: LunarLander-v3 Environment

Overview:
The goal is to land a lunar lander spacecraft safely on a landing pad.

Game Description:
- Lander starts at top center of screen
- Landing pad is at bottom center between two flags
- Must land safely: both legs touching ground, minimal velocity
- Fuel is limited (engines cost fuel)
- Episode ends when: landed, crashed, or timeout (1000 steps)

Physics:
- Gravity pulls lander down
- 4 engines for control:
  * Main engine (thrust downward)
  * Left engine (thrust right)
  * Right engine (thrust left)
  * Do nothing (coast)
- Realistic physics simulation using Box2D

Success Criteria:
- Land between flags
- Both legs on ground
- Low velocity (soft landing)
- Not tipped over
- Bonus for fuel efficiency

Version Note:
- Using LunarLander-v3 (latest version)
- Previous v2 is deprecated
"""

print("\n🔧 Creating LunarLander Environment:")
print("=" * 80)

try:
    # Create environment (v3 is the latest)
    env = gym.make('LunarLander-v3')
    print("✅ LunarLander-v3 environment created successfully!")
    
    # Get environment info
    print("\n📊 Environment Information:")
    print("=" * 80)
    print(f"   Observation space: {env.observation_space}")
    print(f"   Action space: {env.action_space}")
    print(f"   Max episode steps: {env.spec.max_episode_steps if env.spec else 'N/A'}")
    print(f"   Reward threshold: {env.spec.reward_threshold if env.spec else 'N/A'}")
    
    # Test reset
    initial_state, info = env.reset(seed=42)
    print(f"\n   Initial state shape: {initial_state.shape}")
    print(f"   Initial state: {initial_state}")
    
    env.close()
    
except Exception as e:
    print(f"❌ Error creating environment: {e}")
    print("\nTroubleshooting:")
    print("   1. Make sure gymnasium[box2d] is installed")
    print("   2. Try: pip install swig")
    print("   3. Try: pip install box2d-py")
    print("   4. Try: pip install gymnasium[box2d] --upgrade")
    print("   5. Restart kernel if needed")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: Setting Up LunarLander Environment

🔧 Creating LunarLander Environment:
❌ Error creating environment: Box2D is not installed, you can install it by run `pip install swig` followed by `pip install "gymnasium[box2d]"`

Troubleshooting:
   1. Make sure gymnasium[box2d] is installed
   2. Try: pip install swig
   3. Try: pip install box2d-py
   4. Try: pip install gymnasium[box2d] --upgrade
   5. Restart kernel if needed

✅ Exercise 1.1 Complete!
